In [ ]:
!nvidia-smi

In [ ]:
!pip install torch torchvision torchaudio
!pip install tokenizers datasets
!pip install wandb

In [ ]:
import os
os.makedirs("nexus-slm/model",exist_ok = True)
os.makedirs("nexus-slm/data",exist_ok = True)
os.makedirs("nexus-slm/train",exist_ok = True)
os.makedirs("nexus-slm/utils",exist_ok = True)

In [ ]:
from datasets import load_dataset
dataset =  load_dataset("wikitext","wikitext-103-raw-v1")
print(dataset)

In [ ]:
print(dataset["train"][10]["text"][:1000])

In [ ]:
import re
def clean_data(example):
  text = example["text"]
  text = text.strip()
  text = re.sub(r"\sub+","",text)
  return {"text" : text}
dataset = dataset.map(clean_data)


In [ ]:
dataset = dataset.filter(lambda x : len(x["text"]) > 50)

In [ ]:
with open("corpus.txt","w",encoding = "utf-8") as f:
  for example in dataset["train"]:
    f.write(example["text"] + "\n")

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
tokenizer = Tokenizer(BPE(unk_token = "[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size = 24000,special_tokens = ["[POS]","[EOS]","[UNK]","[ESU]"])
tokenizer.train(["corpus.txt"],trainer)


In [ ]:
tokenizer.save("nexus-slm-tokenizer.json")

In [ ]:
enc = tokenizer.encode("Transformers are really powerful models")
print(enc.tokens)
print(enc.ids)

In [ ]:
import random
import numpy as np
def avg_tokens_per_sample(dataset,tokenizer,n=1000):
  lengths = []
  for i in random.sample(range(len(dataset)),n):
    text = dataset[i]["text"]
    enc = tokenizer.encode("text")
    lengths.append(len(enc.ids))
  return np.mean(lengths)
avg_tokens = avg_tokens_per_sample(dataset["train"],tokenizer)
print(f"The avg tokens per sample are:{avg_tokens}")

In [ ]:
print(len(dataset["train"]))
print(len(dataset["train"][0]["text"]))

In [ ]:
enc = tokenizer.encode(dataset["train"][0]["text"])
print(len(enc.ids))
print(enc.tokens[:20])

In [ ]:
def debug_avg(dataset, tokenizer, n=10):
    lengths = []
    for i in range(n):
        text = dataset[i]["text"]
        enc = tokenizer.encode(text)
        lengths.append(len(enc.ids))
        print(f"Sample {i}: {len(enc.ids)} tokens")
    print("Mean:", sum(lengths)/len(lengths))

debug_avg(dataset["train"], tokenizer)

In [ ]:
import torch
import torch.nn as nn
import math

class Config:
  def __init__(self):
    self.vocab_size = 24000
    self.d_model = 512
    self.n_heads = 8
    self.n_layers = 8
    self.d_ff = 4 * self.d_model
    self.max_seq_len = 256
    self.dropout = 0.1


config = Config()

In [ ]:
class RMSnorm(nn.Module):
  def __init__(self,dim,eps=1e-8):
    super().__init__()
    self.eps = eps
    self.scale = nn.Parameter(torch.ones(dim))
    def forward(self,x):
      norm = x.pow(2).mean(-1,keepdim=True)
      x = x * torch.rsqrt(norm + self.eps)
      return self.scale*x

In [ ]:
def rotate_half(x):
  x1 = x[...,:x.shape[-1]//2]
  x2 = x[...,x.shape[-1]//2:]
  return torch.cat((-x2,x1),dim=-1)

def apply_rope(q,k):
  dim = q.shape[-1]
  theta = 10000 * -(torch.arange(0,-1).float()/dim)
  seq_len = q.shape[-2]
  positions  = torch.arange(seq_lean).float()
  freq =  torch.einsum("i,j->ij",positions,theta)
  emb = torch.cat(freq,freq).to(q.device)
  cos = emb.cos[None,None,:,:]
  sin = emb.sin[None,None,:,:]
  q = cos * q + rotate_half(q) * sin
  k = cos * k + rotate_half(k) * sin
  return q,k

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads

        self.qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.proj = nn.Linear(config.d_model, config.d_model)

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        mask = torch.tril(torch.ones(T, T, device=x.device))
        att = att.masked_fill(mask == 0, float('-inf'))

        att = torch.softmax(att, dim=-1)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.proj(out)

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.w1 = nn.Linear(config.d_model, config.d_ff)
        self.w2 = nn.Linear(config.d_model, config.d_ff)
        self.w_out = nn.Linear(config.d_ff, config.d_model)

    def forward(self, x):
        return self.w_out(torch.nn.functional.silu(self.w1(x)) * self.w2(x))

In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(norm + self.eps)
        return self.scale * x

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.norm1 = RMSnorm(config.d_model)
    self.norm2 = RMSnorm(config.d_model)
    self.mlp   = SwiGLU(config)
    self.att   = MultiHeadAttention(config)

    self.gate_att = nn.Parameter(torch.tensor(0.1))
    self.gate_mlp = nn.Parameter(torch.tensor(0.1))

  def forward(self,x):
    x = x + self.gate_att * self.att(self.norm1)
    x = x + self.gate_mlp * self.mlp(self.mlp)
    return x

In [ ]:
class NexusSLM(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(config) for _ in range(config.n_layers)
        ])

        self.norm = RMSNorm(config.d_model)

        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        # weight tying
        self.lm_head.weight = self.token_emb.weight

    def forward(self, input_ids):

        x = self.token_emb(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)        # <- THIS MUST HAVE (x)

        logits = self.lm_head(x)

        return logits           # <- MUST RETURN logits

In [ ]:
model = NexusSLM(config).cuda()

dummy_input = torch.randint(0, config.vocab_size, (2, 64)).cuda()

out = model(dummy_input)

print(out.shape)

In [ ]:
-import torch

def tokenize_function(example):
    tokens = tokenizer.encode(example["text"]).ids
    return {"ids": tokens}

tokenized = dataset["train"].map(tokenize_function, remove_columns=["text"])

In [ ]:
all_tokens = []
for item in tokenized:
    all_tokens.extend(item["ids"])

all_tokens = torch.tensor(all_tokens, dtype=torch.long)
print("Total tokens:", len(all_tokens))

In [ ]:
seq_len = config.max_seq_len

def get_batch(tokens, batch_size):
    ix = torch.randint(0, len(tokens) - seq_len - 1, (batch_size,))
    x = torch.stack([tokens[i:i+seq_len] for i in ix])
    y = torch.stack([tokens[i+1:i+seq_len+1] for i in ix])
    return x.cuda(), y.cuda()

In [ ]:
optimizer = torch.optim.AdamW(.NexusSLM.parameters(), lr=3e-4)

criterion = nn.CrossEntropyLoss()

In [ ]:
scaler = torch.cuda.amp.GradScaler()

In [ ]:
model.train()

x, y = get_batch(all_tokens, batch_size=4)

for step in range(200):
    optimizer.zero_grad()

    with torch.cuda.amp.autocast():
        logits = model(x)
        loss = criterion(logits.view(-1, config.vocab_size), y.view(-1))

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    if step % 20 == 0:
        print(f"Step {step} | Loss: {loss.item()}")

In [ ]:
split_idx = int(0.9 * len(all_tokens))
train_tokens = all_tokens[:split_idx]
val_tokens = all_tokens[split_idx:]

print("Train tokens:", len(train_tokens))
print("Val tokens:", len(val_tokens))

In [ ]:
@torch.no_grad()
def estimate_loss(tokens, eval_iters=20):
    model.eval()
    losses = []

    for _ in range(eval_iters):
        x, y = get_batch(tokens, batch_size=4)
        with torch.cuda.amp.autocast():
            logits = model(x)
            loss = criterion(
                logits.view(-1, config.vocab_size),
                y.view(-1)
            )
        losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)

In [ ]:
max_steps = 2000
eval_interval = 200

for step in range(max_steps):

    x, y = get_batch(train_tokens, batch_size=8)

    optimizer.zero_grad()

    with torch.cuda.amp.autocast():
        logits = model(x)
        loss = criterion(
            logits.view(-1, config.vocab_size),
            y.view(-1)
        )

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    if step % eval_interval == 0:
        train_loss = loss.item()
        val_loss = estimate_loss(val_tokens)

        print(f"Step {step}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss:   {val_loss:.4f}")
        print("-" * 30)

In [ ]:
max_steps = 2000
eval_interval = 200

for step in range(max_steps):

    x, y = get_batch(train_tokens, batch_size=8)

    optimizer.zero_grad()

    with torch.cuda.amp.autocast():
        logits = model(x)
        loss = criterion(
            logits.view(-1, config.vocab_size),
            y.view(-1)
        )

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    if step % eval_interval == 0:
        train_loss = loss.item()
        val_loss = estimate_loss(val_tokens)

        print(f"Step {step}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss:   {val_loss:.4f}")
        print("-" * 30)

        torch.save(model.state_dict(), "nexus_slm_checkpoint.pt")

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=50):
    model.eval()

    input_ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).cuda()

    for _ in range(max_new_tokens):
        input_ids = input_ids[:, -config.max_seq_len:]
        logits = model(input_ids)
        next_token_logits = logits[:, -1, :]
        probs = torch.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=1)

    output = tokenizer.decode(input_ids[0].tolist())
    return output

In [ ]:
print(generate(model, "Artificial intelligence is"))

In [ ]:
temperature = 0.8
probs = torch.softmax(next_token_logits / temperature, dim=-1)

In [ ]:
# after tokenizing dataset
all_tokens = []

for item in tokenized:
    all_tokens.extend(item["ids"])

all_tokens = torch.tensor(all_tokens, dtype=torch.long)
torch.save(all_tokens, "tokens.pt")

print("Total tokens:", len(all_tokens))

In [ ]:
import torch
from config import Config
from model import build_model
from tokenizers import Tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load config + model
config = Config()
model = build_model(config).to(device)
model.load_state_dict(torch.load("checkpoints/nexus_slm.pt", map_location=device))
model.eval()

# Load tokenizer
tokenizer = Tokenizer.from_file("tokenizer.json")


@torch.no_grad()
def generate(prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    input_ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(input_ids).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        input_ids = input_ids[:, -config.max_seq_len:]

        logits = model(input_ids)
        next_token_logits = logits[:, -1, :] / temperature

        # Top-k sampling
        if top_k is not None:
            values, indices = torch.topk(next_token_logits, top_k)
            probs = torch.zeros_like(next_token_logits)
            probs.scatter_(1, indices, torch.softmax(values, dim=-1))
        else:
            probs = torch.softmax(next_token_logits, dim=-1)

        next_token = torch.multinomial(probs, 1)
        input_ids = torch.cat([input_ids, next_token], dim=1)

    return tokenizer.decode(input_ids[0].tolist())

while True:
    user_input = input("\nYou: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    output = generate(user_input)
    print("\nModel:", output)

In [ ]:
!pip install nbstripout